# Your first project — Classifying breast tumors

**Chapter 5 · Unit 1 — Machine Learning Based Data Analysis**

This is your first end-to-end machine learning project. You'll take a real medical dataset and build a classifier that predicts whether a tumor is **malignant** (cancerous) or **benign** (non-cancerous) from measurements made on a digital image of the tumor's cells.

The goal isn't to build the best possible model — it's to walk through the **complete workflow** that every supervised learning project follows:

1. **Load** and **explore** the data
2. **Split** it into training and test sets
3. **Train** a model on the training set
4. **Evaluate** it on the held-out test set
5. **Reflect** on what could be improved

Every project in this chapter and beyond follows this same skeleton. Once you know it, the only thing that changes is which model and which dataset.

## Color key

We'll use color-coded boxes throughout the notebook:

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 8px 12px; margin: 4px 0;"><strong>Question for you</strong> — short conceptual questions to think about.</div>
<div style="background-color: #ede9fe; border-left: 4px solid #7c3aed; padding: 8px 12px; margin: 4px 0;"><strong>Answer</strong> — solutions appear in the instructor version of this notebook.</div>
<div style="background-color: #dbeafe; border-left: 4px solid #3b82f6; padding: 8px 12px; margin: 4px 0;"><strong>Tip</strong> — useful clarifications.</div>
<div style="background-color: #fee2e2; border-left: 4px solid #ef4444; padding: 8px 12px; margin: 4px 0;"><strong>Watch out</strong> — common mistakes to avoid.</div>


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets         import load_breast_cancer
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler
from sklearn.tree             import DecisionTreeClassifier
from sklearn.metrics          import accuracy_score

# A fixed random seed makes the notebook reproducible — your numbers will match ours.
RANDOM_STATE = 42

## 1. Load the data

scikit-learn ships several "toy" datasets bundled with the library — meaning they're available offline and load instantly. The breast cancer dataset is one of them.

In [ ]:
data = load_breast_cancer()

# What did we get back?
print("Keys in the dataset bundle:")
print(list(data.keys()))

<div style="background-color: #dbeafe; border-left: 4px solid #3b82f6; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Tip</strong><br><code>data</code> is a <em>Bunch</em> object — basically a dictionary with attribute access. <code>data.data</code> contains the features (one row per tumor), <code>data.target</code> contains the labels, and <code>data.DESCR</code> contains a human-readable description of the dataset.</div>

In [ ]:
# Peek at the description (just the first 20 lines)
print("\n".join(data.DESCR.split("\n")[:20]))

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>Look at what was just printed and answer:<br>(a) How many samples (tumors) are in the dataset, and how many features (measurements) does each have?<br>(b) Which class is encoded as <code>0</code> and which as <code>1</code>?</div>

In [ ]:
# Build a tidy DataFrame for the rest of the notebook
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="diagnosis")

print(f"Feature matrix X: {X.shape[0]} rows × {X.shape[1]} columns")
print(f"Target vector y: {y.shape[0]} values\n")
print("First 3 rows of X (showing only 5 of the 30 columns):")
print(X.iloc[:3, :5].round(2))

## 2. How balanced are the classes?

Always check class balance before building a classifier. Imbalanced data needs special handling — and even mildly imbalanced data affects how we should think about accuracy.

In [ ]:
class_counts = (y.value_counts()
                  .sort_index()
                  .rename(index={0: "malignant (0)", 1: "benign (1)"}))

print(class_counts)
print(f"\nProportions:\n{(class_counts / class_counts.sum()).round(3)}")

ax = class_counts.plot(kind="bar", color=["#D85A30", "#1D9E75"], rot=0)
ax.set_title("How many tumors of each class?")
ax.set_ylabel("Number of tumors")
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

## 3. What do the features look like?

Before training any model, it's worth peeking at how the two classes differ in feature space. If a single feature already separates the classes nicely, the problem is easier than if every feature overlaps heavily.

In [ ]:
features_to_plot = ["mean radius", "mean concave points"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, feat in zip(axes, features_to_plot):
    ax.hist(X.loc[y == 0, feat], bins=25, alpha=0.7, color="#D85A30",
            label="malignant", edgecolor="white")
    ax.hist(X.loc[y == 1, feat], bins=25, alpha=0.7, color="#1D9E75",
            label="benign",    edgecolor="white")
    ax.set_xlabel(feat)
    ax.set_ylabel("Number of tumors")
    ax.legend()

plt.suptitle("Feature distribution by class", y=1.02)
plt.tight_layout()
plt.show()

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>Looking at the two histograms above, for which feature is it <strong>easier</strong> to tell the two classes apart? What visual cue tells you so?</div>

## 4. Split the data into training and test sets

We never train a model and evaluate it on the same data — that would be like taking an exam where you've already seen the answers. Instead, we hold out part of the data as a **test set** that the model never sees during training, so we can estimate how well it would perform on new patients.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,        # 25% goes to the test set
    stratify=y,            # preserve the malignant/benign ratio in both splits
    random_state=RANDOM_STATE,
)

print(f"Training set: {len(X_train)} tumors")
print(f"Test set:     {len(X_test)} tumors")
print(f"\nClass proportions in TRAIN: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Class proportions in TEST:  {y_test.value_counts(normalize=True).round(3).to_dict()}")

<div style="background-color: #dbeafe; border-left: 4px solid #3b82f6; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Tip</strong><br>The <code>stratify=y</code> argument tells <code>train_test_split</code> to keep the same proportion of each class in both the training and test sets. Without it, a random split could (by bad luck) put more malignant tumors in one set than the other.</div>

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>Why does stratification matter especially when classes are imbalanced (37% malignant, 63% benign in our case)?</div>

## 5. Train a Decision Tree classifier

A Decision Tree is one of the simplest classifiers to understand: it asks a sequence of yes/no questions about the features ("is mean radius less than 14.5?") and uses the answers to navigate to a leaf, where each leaf has a predicted class.

We'll fit one with default settings — it will figure out the right questions and thresholds automatically.

In [ ]:
# Step 1: create the model
model = DecisionTreeClassifier(criterion="entropy", random_state=RANDOM_STATE)

# Step 2: fit it to the training data — this is where the learning happens
model.fit(X_train, y_train)

# Step 3: predict on the test set
y_pred = model.predict(X_test)

# Step 4: measure how well the predictions match the actual labels
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_accuracy:.4f}")

<div style="background-color: #dbeafe; border-left: 4px solid #3b82f6; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Tip</strong><br>The pattern <code>create → fit → predict → score</code> is the same for <em>every</em> scikit-learn model. Decision Trees, k-Nearest Neighbors, Logistic Regression, Random Forest — they all follow this exact four-step recipe. Once you know it, you know how to use any classifier in scikit-learn.</div>

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>We got <strong>~93% accuracy</strong>. That sounds great — but is it really? Think about (a) the always-predict-benign baseline you could compute from the class balance, and (b) the kinds of mistakes the model could be making. What might 93% accuracy be hiding?</div>

## 6. A first look at feature scaling

Many machine learning models work best when all features are on a similar numerical scale. Look at how varied the scales are in our dataset:

In [ ]:
print("Range of each feature (showing the 4 with most extreme ranges):")
ranges = (X.max() - X.min()).sort_values(ascending=False).head(4)
print(ranges.round(2))

**Standardization** is the standard fix: subtract the mean and divide by the standard deviation, so every feature ends up with mean 0 and standard deviation 1. We'll apply it and try two different models on the scaled data — first our Decision Tree, then a new one called K-Nearest Neighbors. The two will react to scaling very differently, and that contrast is the lesson.

<div style="background-color: #fee2e2; border-left: 4px solid #ef4444; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Watch out</strong><br><strong>Always fit the scaler on the training data only.</strong> Use <code>scaler.fit_transform(X_train)</code>, then <code>scaler.transform(X_test)</code>. Never <code>fit_transform</code> on the test set, and never on the combined data — that would let information from the test set leak into your preprocessing, making your evaluation optimistic. We'll come back to this rule throughout the chapter.</div>

In [ ]:
# Standardize features (fit on train only, transform both sets)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### 6.1 — Try the Decision Tree on scaled features

In [ ]:
# Re-train the same Decision Tree on the scaled features
model_scaled = DecisionTreeClassifier(criterion="entropy", random_state=RANDOM_STATE)
model_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = model_scaled.predict(X_test_scaled)

print(f"Decision Tree WITHOUT scaling: {test_accuracy:.4f}")
print(f"Decision Tree WITH    scaling: {accuracy_score(y_test, y_pred_scaled):.4f}")

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>Scaling didn't change the Decision Tree's accuracy at all. <strong>Why?</strong><br>Hint: think about what a Decision Tree actually does. It picks a feature and a threshold ("is mean radius < 14.5?"). What happens to that comparison if we shift and rescale every feature?</div>

### 6.2 — Try K-Nearest Neighbors on the same scaled features

**K-Nearest Neighbors (KNN)** classifies a new patient by finding the *k* most similar patients in the training data and letting them vote. Two patients are "similar" if their measurements are close in feature space — meaning KNN is fundamentally a **distance-based** model. We'll meet it properly in Unit 5; for now, just use it as a contrast to the Decision Tree.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Train two KNN classifiers — one on raw features, one on scaled features
knn_unscaled = KNeighborsClassifier(n_neighbors=5).fit(X_train,        y_train)
knn_scaled   = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)

acc_knn_unscaled = accuracy_score(y_test, knn_unscaled.predict(X_test))
acc_knn_scaled   = accuracy_score(y_test, knn_scaled.predict(X_test_scaled))

print(f"KNN WITHOUT scaling: {acc_knn_unscaled:.4f}")
print(f"KNN WITH    scaling: {acc_knn_scaled:.4f}")
print(f"Improvement from scaling: {acc_knn_scaled - acc_knn_unscaled:+.4f}")

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>This time, scaling jumps KNN's accuracy by about <strong>5 percentage points</strong> — a huge difference for a single preprocessing step. <strong>Why does scaling help KNN so much, when it didn't help the Decision Tree at all?</strong><br>Hint: KNN measures how 'close' two patients are by computing the <em>Euclidean distance</em> between them in feature space.</div>

## 7. What you did

Congratulations — you've completed your first end-to-end machine learning project! You walked through the full workflow:

| Step | What you did | Tool |
|---|---|---|
| Load | Read a real medical dataset | `load_breast_cancer` |
| Explore | Class balance, feature distributions | `value_counts`, `hist` |
| Split | Held out 25% as a stratified test set | `train_test_split` |
| Train | Fit a Decision Tree on the training set | `DecisionTreeClassifier.fit` |
| Predict | Generated predictions on the test set | `model.predict` |
| Evaluate | Compared predictions to actual labels | `accuracy_score` |
| Compare | Saw scaling matter for KNN but not for the Decision Tree | `StandardScaler` |
| Reflect | Asked what 93% accuracy is hiding, and why scaling helps some models but not others | (the questions above!) |

### Three things to take away

1. **The four-step pattern (`create → fit → predict → score`) works for every scikit-learn model.** Once you know this skeleton, you can use any classifier the library offers.
2. **Accuracy is a starting point, not the whole story.** Especially in high-stakes domains where errors aren't symmetric, a single accuracy number can hide what really matters. We'll fix this in Unit 3.
3. **Tree-based models don't need feature scaling — distance-based and margin-based models do.** Decision Tree was unaffected by standardization; KNN gained 5 percentage points. When you meet a new algorithm, the first question to ask is: does it depend on feature scale?

### What's next

The next unit moves to **regression** (predicting a continuous number rather than a class). The workflow stays exactly the same — only the model and the metric change.
